# Distances & Divergences — Minimal Examples

A minimal, runnable example for each distance/divergence in the post
*Taxonomy of Principal Distances & Divergences*. Every cell uses the same small
inputs as the **Example:** lines in the HTML post, so the printed numbers match.

Only `numpy` is required. A few objects (Riemannian/Fisher–Rao geodesics, Finsler
metric, Gromov–Hausdorff, Lévy–Prokhorov) have no simple closed-form code; those are
described in markdown with a simplified note instead of a misleading exact value.

In [ ]:
import numpy as np
np.set_printoptions(precision=4)

## 1. Vector & metric distances
Points $\mathbf p,\mathbf q\in\mathbb R^n$. All are true metrics.

In [ ]:
# Euclidean (L2), Manhattan (L1), Minkowski (Lk), Chebyshev (Linf)
p = np.array([0.0, 0.0]); q = np.array([3.0, 4.0])
print("Euclidean L2:", np.sqrt(((p - q) ** 2).sum()))          # 5.0
print("Manhattan L1:", np.abs(p - q).sum())                    # 7.0
print("Minkowski k=3:", (np.abs(p - q) ** 3).sum() ** (1 / 3)) # ~4.498
print("Chebyshev Linf:", np.abs(p - q).max())                  # 4.0

In [ ]:
# Quadratic (generalized) distance with PSD matrix Q
Q = np.array([[2.0, 0.5], [0.5, 1.0]])
d = p - q
print("Quadratic:", np.sqrt(d @ Q @ d))                        # ~6.782

In [ ]:
# Mahalanobis distance with covariance Sigma
a = np.array([2.0, 1.0]); b = np.array([0.0, 0.0])
Sigma = np.array([[2.0, 0.5], [0.5, 1.0]])
diff = a - b
print("Mahalanobis:", np.sqrt(diff @ np.linalg.inv(Sigma) @ diff))  # ~1.512

In [ ]:
# Hamming distance between two equal-length strings
s1, s2 = "10110", "10011"
print("Hamming:", sum(c1 != c2 for c1, c2 in zip(s1, s2)))     # 2

**String / time-series distances.** Levenshtein (edit distance) and Dynamic Time
Warping are alignment-based; minimal DP implementations below.

In [ ]:
# Levenshtein edit distance (DP) and Dynamic Time Warping (DP)
def levenshtein(s, t):
    m, n = len(s), len(t)
    dp = np.zeros((m + 1, n + 1), int)
    dp[:, 0] = np.arange(m + 1); dp[0, :] = np.arange(n + 1)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s[i-1] == t[j-1] else 1
            dp[i, j] = min(dp[i-1, j] + 1, dp[i, j-1] + 1, dp[i-1, j-1] + cost)
    return dp[m, n]

def dtw(x, y):
    n, m = len(x), len(y)
    D = np.full((n + 1, m + 1), np.inf); D[0, 0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            c = abs(x[i-1] - y[j-1])
            D[i, j] = c + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    return D[n, m]

print("Levenshtein('kitten','sitting'):", levenshtein("kitten", "sitting"))  # 3
print("DTW([1,2,3],[1,2,2,3]):", dtw([1,2,3],[1,2,2,3]))                      # 0.0

## 2. Riemannian & information geometry
Distance becomes the length of the shortest path (geodesic) on a curved manifold;
for statistical models the curvature is the Fisher information.

In [ ]:
# Fisher information of a Gaussian N(mu, sigma^2): I(mu) = 1 / sigma^2 (analytic),
# verified numerically as E[(d/dmu log p)^2].
sigma = 2.0
rng = np.random.default_rng(0)
x = rng.normal(0.0, sigma, size=200000)
score = x / sigma**2                       # d/dmu log p at mu=0
print("Fisher info I(mu) analytic:", 1 / sigma**2)   # 0.25
print("Fisher info I(mu) numeric :", (score**2).mean())

**Riemannian geodesic, Finsler metric, Fisher–Rao distance** — no simple
closed-form snippet (they require solving a variational/geodesic problem on a manifold).
Locally the Fisher–Rao distance satisfies $\rho_{FR}(p,q)\approx\sqrt{2\,\mathrm{KL}(p\|q)}$,
so for nearby distributions one can approximate it from the KL value computed below.

## 3a. $f$-divergences (Csiszár / Ali–Silvey)
Template $D_f(p\|q)=\int p\,f(q/p)\,d\mu$. Examples on the pmfs
$p=[0.5,0.5]$, $q=[0.9,0.1]$.

In [ ]:
p = np.array([0.5, 0.5]); q = np.array([0.9, 0.1])
print("KL(p||q) nats :", (p * np.log(p / q)).sum())            # ~0.511
print("Pearson chi2  :", (((q - p) ** 2) / p).sum())           # 0.64
print("Hellinger     :", np.sqrt(((np.sqrt(p) - np.sqrt(q)) ** 2).sum() / 2))  # ~0.325
print("Total variation:", 0.5 * np.abs(p - q).sum())           # 0.4

# Amari alpha-divergence, alpha = 0  ->  4 (1 - sum sqrt(p q))
alpha = 0.0
t = q / p
f = 4 / (1 - alpha**2) * (1 - t ** ((1 + alpha) / 2))
print("Amari alpha=0 :", (p * f).sum())                        # ~0.422

## 3b. Bregman divergences
Template $B_F(x\|y)=F(x)-F(y)-\langle x-y,\nabla F(y)\rangle$.

In [ ]:
# Squared Euclidean (F = ||.||^2)
x = np.array([1.0, 2.0]); y = np.array([0.0, 0.0])
print("Bregman sq-Euclidean:", ((x - y) ** 2).sum())           # 5.0

# Itakura-Saito (F = -sum log x_i)
pi = np.array([1.0, 2.0, 3.0]); qi = np.array([1.0, 1.0, 4.0])
print("Itakura-Saito:", (pi / qi - np.log(pi / qi) - 1).sum()) # ~0.345

# Log-Det divergence on SPD matrices
P = np.array([[2.0, 0.0], [0.0, 1.0]]); Qm = np.eye(2); n = 2
PQ = P @ np.linalg.inv(Qm)
print("Log-Det:", np.trace(PQ) - np.log(np.linalg.det(PQ)) - n)  # ~0.307

## 3c. Overlap / $\alpha$-power family
All built from the affinity $\int p^{\alpha}q^{1-\alpha}d\mu$. Same $p,q$ pmfs.

In [ ]:
p = np.array([0.5, 0.5]); q = np.array([0.9, 0.1])
BC = np.sqrt(p * q).sum()
print("Bhattacharyya coeff:", BC)                              # ~0.894
print("Bhattacharyya dist :", -np.log(BC))                     # ~0.112

a = 0.5
print("Renyi div a=0.5    :", 1 / (a - 1) * np.log((p**a * q**(1 - a)).sum()))  # ~0.223

als = np.linspace(0.01, 0.99, 99)
chernoff = max(-np.log((p**al * q**(1 - al)).sum()) for al in als)
print("Chernoff info      :", chernoff)                        # ~0.112

## 3d. Symmetrized & Jensen-type

In [ ]:
p = np.array([0.5, 0.5]); q = np.array([0.9, 0.1])
kl = lambda a, b: (a * np.log(a / b)).sum()
print("Jeffreys:", kl(p, q) + kl(q, p))                        # ~0.879

m = (p + q) / 2
js = 0.5 * kl(p, m) + 0.5 * kl(q, m)
print("Jensen-Shannon:", js, " sqrt(JS):", np.sqrt(js))        # ~0.102, ~0.319

# Burbea-Rao with F = negative Shannon entropy  ->  equals JS
negH = lambda z: (z * np.log(z)).sum()
print("Burbea-Rao (=JS):", (negH(p) + negH(q)) / 2 - negH(m))  # ~0.102

## 4. Entropies
Uncertainty of a single distribution. Example pmf $r=[0.5,0.25,0.25]$.

In [ ]:
r = np.array([0.5, 0.25, 0.25])
print("Shannon (nats):", -(r * np.log(r)).sum())               # ~1.040
print("Shannon (bits):", -(r * np.log2(r)).sum())              # 1.5

alpha = 2.0
print("Renyi entropy a=2 :", 1 / (1 - alpha) * np.log((r**alpha).sum()))  # ~0.981
print("Tsallis a=2       :", 1 / (1 - alpha) * ((r**alpha).sum() - 1))    # 0.625

beta = 3.0
sm = 1 / (1 - beta) * ((r**alpha).sum() ** ((1 - beta) / (1 - alpha)) - 1)
print("Sharma-Mittal a=2,b=3:", sm)                            # ~0.430

## 5. Set & metric-space distances

In [ ]:
# Hausdorff distance between two finite point sets
A = np.array([[0, 0], [1, 0], [0, 1.0]]); B = np.array([[0, 0], [1, 1.0]])
def directed(X, Y):
    return max(np.min(np.sqrt(((x - Y) ** 2).sum(1))) for x in X)
print("Hausdorff:", max(directed(A, B), directed(B, A)))       # 1.0

**Gromov–Hausdorff distance** compares whole metric spaces up to isometry; computing
it exactly is NP-hard (an infimum over all isometric embeddings), so no minimal snippet is
given. In practice it is approximated (e.g. via the Gromov–Wasserstein relaxation).

## 6. Optimal transport & integral probability metrics (IPMs)

In [ ]:
# Wasserstein-1 between two equal-size 1-D samples = mean |sorted differences|
u = np.array([0.0, 1, 2, 3]); v = np.array([1.0, 2, 3, 4])
print("Wasserstein-1:", np.abs(np.sort(u) - np.sort(v)).mean())  # 1.0

# Kolmogorov-Smirnov statistic = sup |F_u - F_v| on the pooled grid
grid = np.sort(np.concatenate([u, v]))
Fu = np.searchsorted(np.sort(u), grid, side="right") / len(u)
Fv = np.searchsorted(np.sort(v), grid, side="right") / len(v)
print("KS statistic:", np.abs(Fu - Fv).max())                  # 0.25

# Maximum Mean Discrepancy (squared, biased) with an RBF kernel
def mmd2(X, Y, g=0.5):
    XX = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
    YY = ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
    XY = ((X[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
    return np.exp(-g * XX).mean() + np.exp(-g * YY).mean() - 2 * np.exp(-g * XY).mean()
X = np.array([[0.0], [1.0], [2.0]]); Y = np.array([[3.0], [4.0], [5.0]])
print("MMD^2 (RBF, g=0.5):", mmd2(X, Y))                       # ~1.063

**Lévy–Prokhorov distance** (metrizes weak convergence) and **Stein discrepancy**
(needs a Stein operator / score) have no one-line example; LP is an infimum over all Borel
sets and is typically only bounded, not computed directly.

## 7. Quantum geometry (density matrices)
Densities become density matrices $\rho$; integrals become traces. With diagonal
$\rho$ these reduce to the classical entropy/KL on the eigenvalues.

In [ ]:
# Von Neumann entropy of a density matrix (diagonal here)
rho = np.array([[0.7, 0.0], [0.0, 0.3]])
ev = np.linalg.eigvalsh(rho)
print("Von Neumann entropy (nats):", -(ev * np.log(ev)).sum())  # ~0.611

# Von Neumann (quantum relative) divergence Tr(P(logP - logQ) - P + Q)
def logm_sym(M):
    w, V = np.linalg.eigh(M)
    return V @ np.diag(np.log(w)) @ V.T
P = np.array([[0.7, 0.0], [0.0, 0.3]]); Qd = np.array([[0.5, 0.0], [0.0, 0.5]])
vn = np.trace(P @ (logm_sym(P) - logm_sym(Qd)) - P + Qd)
print("Von Neumann divergence (nats):", vn)                    # ~0.082